# Build Gaia DR3 XP Coefficient Dataset

This notebook constructs the base binary-classification dataset from VOSA labels and Gaia DR3 XP continuous mean-spectrum coefficients. It is the first database-construction stage for experiments that use the normalized BP/RP coefficient representation.

Inputs:
- `preparations/VOSA_labels_training.csv`, containing Gaia DR3 source IDs and binary labels. The legacy location `data/VOSA_labels_training.csv` is also accepted.
- Gaia DR3 XP continuous mean-spectrum VOTables, either already cached locally or retrieved through Gaia@AIP.

Primary outputs:
- `out_data/gaia_dr3_xp_c110_l2_binary.csv`
- `out_data/gaia_dr3_xp_c110_binary.npz`

Feature schema:
- `source_id`: Gaia DR3 source identifier.
- `y`: binary class label.
- `c000..c109`: L2-normalized 110-dimensional XP coefficient vector, with BP coefficients followed by RP coefficients.

Credential handling:
- Set `GAIA_AIP_TOKEN` in the environment or in a local `.env` file.
- Alternatively, create `.gaia_aip_token` containing either the raw token or `Token <token>`.
- Tokens are used only for Gaia@AIP requests and are never printed.


In [ ]:
import ast
import io
import os
import re
import shutil
import time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import pyvo as vo
import requests
from astropy.table import Table


## Configuration

Adjust only the paths and chunk sizes in this cell for a different project layout or Gaia@AIP throughput limit.


In [ ]:
ROOT = Path.cwd().resolve()
LABEL_FILE = "VOSA_labels_training.csv"
LABEL_RELATIVE_PATHS = [
    Path("preparations") / LABEL_FILE,
    Path("data") / LABEL_FILE,
]

search_roots = [ROOT, *ROOT.parents]
label_matches = []
for candidate_root in search_roots:
    candidate_roots = []
    packaged_root = candidate_root / "prep"
    if packaged_root.exists():
        candidate_roots.append(packaged_root)
    candidate_roots.append(candidate_root)

    for base_root in candidate_roots:
        for relative_path in LABEL_RELATIVE_PATHS:
            candidate = base_root / relative_path
            if candidate.exists():
                label_matches = [candidate.resolve()]
                break
        if label_matches:
            break
    if label_matches:
        break

if not label_matches:
    allowed_parent_dirs = {"preparations", "data"}
    label_matches = [
        path.resolve()
        for path in ROOT.rglob(LABEL_FILE)
        if path.parent.name in allowed_parent_dirs
    ]

if not label_matches:
    expected_locations = ", ".join(str(path) for path in LABEL_RELATIVE_PATHS)
    raise FileNotFoundError(
        f"Could not find {LABEL_FILE}. Place the VOSA label file under one of: {expected_locations}."
    )

LABEL_PATH = label_matches[0]
ROOT = LABEL_PATH.parents[1].resolve()
LABEL_DIR = LABEL_PATH.parent
OUT_DIR = ROOT / "out_data"
CACHE_DIR = OUT_DIR / "gaia_aip_cache" / "xp_continuous"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_CSV = OUT_DIR / "gaia_dr3_xp_c110_l2_binary.csv"
CANONICAL_NPZ = OUT_DIR / "gaia_dr3_xp_c110_binary.npz"

WRITE_COMPATIBILITY_ALIASES = True
LEGACY_CSV_ALIASES = [
    OUT_DIR / "binary_vosa_XPcoeff_110d_L2.csv",
    ROOT / "binary_vosa_XPcoeff_110d_L2.csv",
]
LEGACY_NPZ_ALIASES = [OUT_DIR / "binary_vosa_XPcoeff_110d.npz"]

TAP_URL = "https://gaia.aip.de/tap"
SJS_URL = "https://gaia.aip.de/uws/simple-join-service"
XP_JOIN_TABLE = "gaiadr3.xp_continuous_mean_spectrum"

ID_CHUNK_SIZE = 400
REQUIRE_ALL_LABELS_HAVE_XP = True

print("ROOT    :", ROOT)
print("LABEL_PATH:", LABEL_PATH)
print("OUT_DIR :", OUT_DIR)


## Shared helpers

These functions handle credentials, Gaia@AIP jobs, array parsing, and compatibility file writing.


In [ ]:
PLACEHOLDER_TOKENS = {"", "YOUR_TOKEN_GOES_HERE", "<YOUR_TOKEN>", "<TOKEN>", "TOKEN"}


def load_dotenv_if_available() -> None:
    try:
        from dotenv import load_dotenv
    except Exception:
        return
    load_dotenv(ROOT / ".env")
    load_dotenv()


def normalize_gaia_aip_token(raw_token: str) -> str:
    token = (raw_token or "").strip()
    if len(token) >= 2 and token[0] in {"'", '"'} and token[-1] == token[0]:
        token = token[1:-1].strip()
    if token.upper() in PLACEHOLDER_TOKENS:
        raise RuntimeError(
            "Gaia@AIP token is missing. Set GAIA_AIP_TOKEN, define it in .env, "
            "or create .gaia_aip_token with your Gaia@AIP API token."
        )
    if any(ord(char) > 127 for char in token):
        raise RuntimeError("GAIA_AIP_TOKEN must contain ASCII characters only. Check for copied whitespace or quotes.")
    return token if token.startswith("Token ") else f"Token {token}"


def load_gaia_aip_token(required: bool = True) -> str | None:
    load_dotenv_if_available()
    raw_token = os.getenv("GAIA_AIP_TOKEN", "").strip()
    token_file = Path(os.getenv("GAIA_AIP_TOKEN_FILE", str(ROOT / ".gaia_aip_token"))).expanduser()

    if not raw_token and token_file.exists():
        raw_token = token_file.read_text(encoding="utf-8").strip()

    if not raw_token and not required:
        return None
    return normalize_gaia_aip_token(raw_token)


def make_gaia_aip_session(required: bool = True) -> requests.Session | None:
    token = load_gaia_aip_token(required=required)
    if token is None:
        return None
    session = requests.Session()
    session.headers["Authorization"] = token
    return session


def raise_for_aip_response(response: requests.Response, context: str) -> None:
    text = response.text or ""
    if response.status_code in {401, 403}:
        raise RuntimeError(
            f"Gaia@AIP rejected {context} with HTTP {response.status_code}. "
            "Check GAIA_AIP_TOKEN. The value may be either the raw token or 'Token <token>'. "
            "Restart the kernel after changing credentials."
        )
    response.raise_for_status()


def tap_create_async_job(session: requests.Session, query: str, runid: str) -> str:
    payload = {
        "REQUEST": "doQuery",
        "LANG": "ADQL",
        "FORMAT": "votable",
        "QUERY": query.strip().rstrip(";"),
        "RUNID": runid,
    }
    response = session.post(f"{TAP_URL}/async", data=payload, allow_redirects=False, timeout=120)
    if response.status_code in {302, 303} and "Location" in response.headers:
        job_url = response.headers["Location"]
        if job_url.startswith("/"):
            job_url = "https://gaia.aip.de" + job_url
    else:
        raise_for_aip_response(response, "TAP async job creation")
        raise RuntimeError(
            f"Unexpected Gaia@AIP TAP response for {runid}: HTTP {response.status_code}; "
            f"body={response.text[:300]!r}"
        )
    session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, timeout=60).raise_for_status()
    return job_url


def wait_for_uws_job(session: requests.Session, job_url: str, *, timeout_s: int, label: str) -> None:
    started = time.time()
    while True:
        phase = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if phase in {"COMPLETED", "ERROR", "ABORTED"}:
            break
        if time.time() - started > timeout_s:
            raise TimeoutError(f"{label} timed out after {timeout_s} seconds: {job_url}")
        time.sleep(1.5)

    if phase != "COMPLETED":
        details = ""
        if phase == "ERROR":
            try:
                details = session.get(f"{job_url}/error", timeout=60).text[:800]
            except Exception:
                details = ""
        raise RuntimeError(f"{label} ended with phase={phase}. {details}")


def sjs_download(session: requests.Session, tap_job_url: str, join_table: str, output_dir: Path) -> Path:
    service = vo.dal.DALService(SJS_URL, session=session)
    tap_job_id = tap_job_url.rstrip("/").split("/")[-1]
    query = service.create_query(
        job_id=tap_job_id,
        column_name="source_id",
        responseformat="votable",
        join_table=join_table,
        data_structure="COMBINED",
    )
    response = query.submit(post=True)
    raise_for_aip_response(response, "Simple Join Service job creation")

    job = vo.io.uws.parse_job(io.BytesIO(response.content))
    job_url = f"{service._baseurl}/{job.jobid}"
    service._session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, stream=True).raise_for_status()
    wait_for_uws_job(service._session, job_url, timeout_s=300, label="Gaia@AIP Simple Join Service job")

    job_refreshed = vo.io.uws.parse_job(io.BytesIO(service._session.get(job_url, timeout=60).content))
    results = job_refreshed.results
    if hasattr(results, "keys") and callable(results.keys):
        first_key = sorted(results.keys())[0]
        href = results[first_key].href
        result_key = str(first_key)
    else:
        result_list = list(results)
        if not result_list:
            raise RuntimeError("Gaia@AIP Simple Join Service completed but returned no result links.")
        href = result_list[0].href
        result_key = "result"

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"sjs_{job_refreshed.jobid}_{result_key}.vot"
    last_content = b""
    for attempt in range(1, 5):
        result_response = service._session.get(href, timeout=300)
        raise_for_aip_response(result_response, "Simple Join Service result download")
        last_content = result_response.content
        looks_complete = (
            len(last_content) > 500
            and b"VOTABLE" in last_content[:5000]
            and b"</VOTABLE>" in last_content[-5000:]
        )
        if looks_complete:
            output_path.write_bytes(last_content)
            return output_path
        time.sleep(float(attempt))

    output_path.write_bytes(last_content)
    raise RuntimeError(
        "Gaia@AIP Simple Join Service result did not look like a complete VOTable. "
        f"The last payload was saved to {output_path}. Re-run this cell after checking the service status."
    )


def to_1d_float_array(value, expected_len: int | None = None) -> np.ndarray:
    if isinstance(value, np.ma.MaskedArray):
        value = np.ma.getdata(value)
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    if isinstance(value, str):
        text = value.strip()
        try:
            value = ast.literal_eval(text)
        except Exception:
            text = text.strip("[]()")
            value = [part for part in re.split(r"[,\s]+", text) if part]
    array = np.asarray(value, dtype=float).reshape(-1)
    if expected_len is not None and array.size != expected_len:
        raise ValueError(f"Expected an array of length {expected_len}, received length {array.size}.")
    return array


def pad_or_truncate(array, length: int = 55) -> np.ndarray:
    values = np.asarray(array, dtype=float).reshape(-1)
    if values.size == length:
        return values
    if values.size > length:
        return values[:length]
    padded = np.zeros(length, dtype=float)
    padded[: values.size] = values
    return padded


def write_aliases(canonical_path: Path, aliases: Iterable[Path]) -> list[Path]:
    written = []
    for alias in aliases:
        alias = Path(alias)
        if alias.resolve() == canonical_path.resolve():
            continue
        alias.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(canonical_path, alias)
        written.append(alias)
    return written


## Load and validate VOSA labels


In [ ]:
label_path = LABEL_PATH
df_labels = pd.read_csv(label_path)

rename_map = {}
if "GaiaDR3" in df_labels.columns and "source_id" not in df_labels.columns:
    rename_map["GaiaDR3"] = "source_id"
if "VOSA" in df_labels.columns and "label_raw" not in df_labels.columns:
    rename_map["VOSA"] = "label_raw"
if rename_map:
    df_labels = df_labels.rename(columns=rename_map)

required_label_columns = {"source_id", "label_raw"}
missing_label_columns = required_label_columns - set(df_labels.columns)
if missing_label_columns:
    raise KeyError(
        f"{label_path} is missing required columns {sorted(missing_label_columns)}. "
        f"Available columns: {list(df_labels.columns)}"
    )

df_labels = df_labels[["source_id", "label_raw"]].dropna().copy()
df_labels["source_id"] = pd.to_numeric(df_labels["source_id"], errors="raise").astype("int64")
df_labels["y"] = pd.to_numeric(df_labels["label_raw"], errors="raise").astype(int)

invalid_labels = sorted(set(df_labels["y"]) - {0, 1})
if invalid_labels:
    raise ValueError(f"Expected binary labels 0/1 only. Found labels: {invalid_labels}")

conflicting_labels = df_labels.groupby("source_id")["y"].nunique()
conflicting_labels = conflicting_labels[conflicting_labels > 1]
if not conflicting_labels.empty:
    raise ValueError(
        "The VOSA label table contains conflicting labels for the same source_id. "
        f"First conflicts: {conflicting_labels.head(10).index.tolist()}"
    )

df_labels = df_labels.drop_duplicates("source_id").sort_values("source_id").reset_index(drop=True)
source_ids = df_labels["source_id"].to_numpy(dtype="int64")

print("Label rows        :", len(df_labels))
print("Unique source_ids :", len(source_ids))
print("Class balance     :", df_labels["y"].value_counts().sort_index().to_dict())


## Load or download Gaia XP coefficient VOTables


In [ ]:
def cached_xp_votables() -> list[Path]:
    canonical_cache = sorted(CACHE_DIR.glob("sjs_*_result.vot"))
    if canonical_cache:
        return canonical_cache
    legacy_cache = sorted(OUT_DIR.glob("sjs_*_result.vot"))
    return legacy_cache


def download_xp_continuous_votables(ids: np.ndarray) -> list[Path]:
    session = make_gaia_aip_session(required=True)
    output_paths = []
    for start in range(0, len(ids), ID_CHUNK_SIZE):
        chunk = ids[start : start + ID_CHUNK_SIZE]
        ids_sql = ",".join(str(int(source_id)) for source_id in chunk)
        query = f"""
        SELECT source_id
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_sql})
        """
        chunk_number = start // ID_CHUNK_SIZE + 1
        print(f"Downloading XP coefficient chunk {chunk_number}: {len(chunk)} source IDs")
        tap_job_url = tap_create_async_job(session, query, runid=f"xp_coefficients_ids_{chunk_number:04d}")
        wait_for_uws_job(session, tap_job_url, timeout_s=240, label="Gaia@AIP TAP source-id job")
        output_path = sjs_download(session, tap_job_url, XP_JOIN_TABLE, CACHE_DIR)
        output_paths.append(output_path)
        print("  saved", output_path)
    return output_paths


votable_paths = cached_xp_votables()
if not votable_paths:
    print("No cached XP coefficient VOTables found; downloading from Gaia@AIP.")
    votable_paths = download_xp_continuous_votables(source_ids)
else:
    print(f"Using {len(votable_paths)} cached XP coefficient VOTables.")

print("First cache file:", votable_paths[0] if votable_paths else "none")


## Build normalized coefficient features


In [ ]:
def ensure_source_id(df: pd.DataFrame, path: Path) -> pd.DataFrame:
    if "source_id" in df.columns:
        out = df.copy()
        out["source_id"] = pd.to_numeric(out["source_id"], errors="raise").astype("int64")
        return out
    if "datalinkID" in df.columns:
        source_id = df["datalinkID"].astype(str).str.extract(r"(\d{10,})")[0]
        if source_id.isna().any():
            raise KeyError(f"Could not parse source_id from datalinkID in {path}.")
        out = df.copy()
        out["source_id"] = source_id.astype("int64")
        return out
    raise KeyError(f"No source_id or datalinkID column found in {path}. Columns: {list(df.columns)}")


xp_tables = []
for votable_path in votable_paths:
    table = Table.read(votable_path)
    xp_tables.append(ensure_source_id(table.to_pandas(), votable_path))

if not xp_tables:
    raise RuntimeError("No XP coefficient tables were loaded.")

df_xp = pd.concat(xp_tables, ignore_index=True)
required_xp_columns = {"source_id", "bp_coefficients", "rp_coefficients"}
missing_xp_columns = required_xp_columns - set(df_xp.columns)
if missing_xp_columns:
    raise KeyError(f"XP data is missing required columns {sorted(missing_xp_columns)}.")

df_xp = df_xp.drop_duplicates("source_id")
df_xp = df_xp[df_xp["source_id"].isin(source_ids)].sort_values("source_id").reset_index(drop=True)

expected_ids = set(int(source_id) for source_id in source_ids)
available_ids = set(int(source_id) for source_id in df_xp["source_id"])
missing_ids = sorted(expected_ids - available_ids)
if missing_ids and REQUIRE_ALL_LABELS_HAVE_XP:
    raise RuntimeError(
        f"XP coefficients are missing for {len(missing_ids)} labeled sources. "
        f"First missing source_ids: {missing_ids[:10]}"
    )

bp = df_xp["bp_coefficients"].map(to_1d_float_array).map(lambda values: pad_or_truncate(values, 55))
rp = df_xp["rp_coefficients"].map(to_1d_float_array).map(lambda values: pad_or_truncate(values, 55))

X_raw = np.hstack([np.vstack(bp.to_numpy()), np.vstack(rp.to_numpy())]).astype(np.float32)
row_norms = np.linalg.norm(X_raw, axis=1, keepdims=True)
zero_norm_rows = np.where(row_norms.reshape(-1) == 0)[0]
if zero_norm_rows.size:
    bad_source_ids = df_xp.iloc[zero_norm_rows]["source_id"].head(10).tolist()
    raise RuntimeError(f"Found zero-norm XP coefficient vectors. First affected source_ids: {bad_source_ids}")

X_l2 = X_raw / row_norms

df_features = pd.DataFrame(X_l2, columns=[f"c{index:03d}" for index in range(X_l2.shape[1])])
df_features.insert(0, "source_id", df_xp["source_id"].to_numpy(dtype="int64"))
df_output = df_labels[["source_id", "y"]].merge(df_features, on="source_id", how="inner")

if len(df_output) != len(source_ids):
    raise RuntimeError(f"Output row count mismatch: expected {len(source_ids)}, received {len(df_output)}.")

print("XP rows used :", len(df_xp))
print("Output shape :", df_output.shape)
df_output.head()


## Save database files


In [ ]:
df_output.to_csv(CANONICAL_CSV, index=False)

np.savez_compressed(
    CANONICAL_NPZ,
    X=X_raw,
    X_l2=X_l2.astype(np.float32),
    y=df_output["y"].to_numpy(dtype=int),
    source_id=df_output["source_id"].to_numpy(dtype="int64"),
)

csv_aliases = write_aliases(CANONICAL_CSV, LEGACY_CSV_ALIASES) if WRITE_COMPATIBILITY_ALIASES else []
npz_aliases = write_aliases(CANONICAL_NPZ, LEGACY_NPZ_ALIASES) if WRITE_COMPATIBILITY_ALIASES else []

print("Canonical CSV:", CANONICAL_CSV)
print("Canonical NPZ:", CANONICAL_NPZ)
if csv_aliases:
    print("CSV compatibility aliases:")
    for path in csv_aliases:
        print(" -", path)
if npz_aliases:
    print("NPZ compatibility aliases:")
    for path in npz_aliases:
        print(" -", path)
